# Exploratory Data Analysis (EDA)
To the data distribution, term frequencies, and identifying potential issues.

In [ ]:

import re
import matplotlib.pyplot as plt
from collections import Counter

import pandas as pd
import openpyxl

# 1. Load the correct Excel file for each company
data_adidas = pd.read_excel("../data/preparation/adidas.xlsx", sheet_name="Sheet1")
data_puma = pd.read_excel("../data/preparation/puma.xlsx", sheet_name="Sheet1")
data_asics = pd.read_excel("../data/preparation/asics.xlsx", sheet_name="Sheet1")
data_mizuno = pd.read_excel("../data/preparation/mizuno.xlsx", sheet_name="Sheet1")

# 2. Store datasets by company
datasets = {
    "adidas": data_adidas,
    "puma": data_puma,
    "asics": data_asics,
    "mizuno": data_mizuno,
}

# 3. Filter text types and calculate the 20 most common words
for company, data in datasets.items():

    # Work on a copy
    df = data.copy()

    # Keep only body_text and question rows
    if "Text_type" in df.columns:
        selected_rows = (
            df["Text_type"]
            .astype("string")
            .str.strip()
            .str.lower()
            .isin(["body_text", "question"])
        )

        df = df.loc[selected_rows].copy()
    else:
        print(f"Warning: Text_type column not found for {company}.")
        continue

    # Check that the Text column exists
    if "Text" not in df.columns:
        print(f"Warning: Text column not found for {company}.")
        continue

    # Combine text, convert to lowercase, and extract words
    text = " ".join(df["Text"].dropna().astype(str))
    all_words = re.findall(r"\b[a-z]+(?:['’][a-z]+)*\b", text.lower())

    # Get the 20 most common words
    common_words = Counter(all_words).most_common(20)

    if not common_words:
        print(f"No words found for {company}.")
        continue

    words, counts = zip(*common_words)

    # 4. Plot the results
    fig, ax = plt.subplots(figsize=(15, 8))

    bars = ax.bar(
        words,
        counts,
        color="steelblue",
        edgecolor="black"
    )

    ax.set_title(
        f"Top 20 Most Common Words — {company.capitalize()}",
        fontsize=14,
        fontweight="bold",
    )

    ax.set_xlabel("Words", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12)

    ax.bar_label(bars, padding=3, fontsize=10)
    ax.margins(y=0.12)
    ax.tick_params(axis="x", rotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")

    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    fig.tight_layout()
    plt.show()
    plt.close(fig)

    print(f"{company.capitalize()}: {len(df)} rows analyzed.")
